In [ ]:
import pandas, driftbench as db
print(db.__version__, pandas.__version__)

In [ ]:
# 1) make sure 0.1.1 is the installed version on disk
import glob, subprocess, os
hits = glob.glob("/content/dc_pkg/drift-conference")
if not hits:
    z = glob.glob("/content/drive/MyDrive/**/drift-conference*.zip", recursive=True)[0]
    subprocess.run(["rm","-rf","/content/dc_pkg"], check=True)
    os.makedirs("/content/dc_pkg", exist_ok=True)
    subprocess.run(["unzip","-q",z,"-d","/content/dc_pkg"], check=True)
subprocess.run(["pip","-q","install","--no-deps","--force-reinstall",
                "/content/dc_pkg/drift-conference"], check=True)

# 2) purge the stale driftbench from memory, then re-import
import sys
for m in list(sys.modules):
    if m == "driftbench" or m.startswith("driftbench."):
        del sys.modules[m]

import driftbench as db
print("driftbench:", db.__version__)
from driftbench.cleaning import clean_csv_streaming
print("clean_csv_streaming imported OK")

In [ ]:
import numpy as np, pandas as pd
from driftbench.cleaning import clean_flows   # this exists in 0.1.0

def clean_csv_streaming(csv_paths, cap_per_class=200_000, chunksize=500_000, seed=0, dedup=True):
    rng = np.random.default_rng(seed)
    # pre-pass: per-class counts (label-col only)
    from collections import Counter
    counts = Counter()
    for p in csv_paths:
        hdr = pd.read_csv(p, nrows=0)
        lc = [c for c in hdr.columns if str(c).strip().lower()=="label"]
        lc = lc[0] if lc else hdr.columns[-1]
        for ch in pd.read_csv(p, usecols=[lc], chunksize=2_000_000, low_memory=False):
            counts.update(ch[lc].astype(str).str.strip().value_counts().to_dict())
    keep = {c: min(1.0, cap_per_class/n) for c,n in counts.items()} if cap_per_class else None

    F,M,L,dd = [],[],[],0
    for p in csv_paths:
        for ch in pd.read_csv(p, chunksize=chunksize, low_memory=False):
            r = clean_flows(ch, drop_constant=False, dedup=dedup)
            dd += r.n_dedup_removed
            f,m,l = r.features, r.metadata, r.labels
            if keep is not None:
                pr = l.map(lambda c: keep.get(c,1.0)).to_numpy(float)
                mask = rng.random(len(pr)) < pr
                f,m,l = f[mask], m[mask], l[mask]
            if len(l): F.append(f); M.append(m); L.append(l)
    feats = pd.concat(F, ignore_index=True); meta = pd.concat(M, ignore_index=True)
    labs = pd.concat(L, ignore_index=True).reset_index(drop=True)
    feats = feats.fillna(feats.median(numeric_only=True))
    const = feats.nunique(); dropped = const[const<=1].index.tolist()
    feats = feats.drop(columns=dropped)
    class R: pass
    r=R(); r.features=feats; r.metadata=meta; r.labels=labs; r.dropped_constant=dropped; r.n_dedup_removed=dd
    return r

print("clean_csv_streaming defined inline OK")

In [ ]:
!pip -q install kaggle
from google.colab import files
files.upload()                      # kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!mkdir -p /content/ddos_tmp
!kaggle datasets download -d rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files -p /content/ddos_tmp --unzip
import glob
print("01-12 files:", sorted(glob.glob('/content/ddos_tmp/01-12/*.csv')))

In [ ]:
import os
os.environ['DRIFT_DATA_ROOT'] = '/content/drive/MyDrive/drift-conference/data'
from pathlib import Path
from driftbench import config

ddos = sorted(Path('/content/ddos_tmp/01-12').glob('*.csv'))
print("2019 files:", len(ddos))
res = clean_csv_streaming(ddos, cap_per_class=200_000, chunksize=500_000, seed=0)
print("features", res.features.shape, "| dropped_const", len(res.dropped_constant))
print("rows per class:", res.labels.value_counts().to_dict())

out = config.INTERIM_DIR / 'CIC-DDoS2019'; out.mkdir(parents=True, exist_ok=True)
res.features.to_parquet(out/'features.parquet')
res.metadata.to_parquet(out/'metadata.parquet')
res.labels.to_frame('label').to_parquet(out/'labels.parquet')
print("written:", out)

In [ ]:
ids2018 = sorted(Path('/content/drive/MyDrive/drift-conference/data/raw/CSE-CIC-IDS2018').glob('*.csv'))
print("2018 files:", len(ids2018))
res = clean_csv_streaming(ids2018, cap_per_class=200_000, chunksize=500_000, seed=0)
print("features", res.features.shape, "| dropped_const", len(res.dropped_constant))
print("rows per class:", res.labels.value_counts().to_dict())

out = config.INTERIM_DIR / 'CSE-CIC-IDS2018'; out.mkdir(parents=True, exist_ok=True)
res.features.to_parquet(out/'features.parquet')
res.metadata.to_parquet(out/'metadata.parquet')
res.labels.to_frame('label').to_parquet(out/'labels.parquet')
print("written:", out)

In [ ]:
import pandas as pd
from pathlib import Path
from driftbench import config

p = config.INTERIM_DIR / 'CSE-CIC-IDS2018'
feats = pd.read_parquet(p/'features.parquet')
meta  = pd.read_parquet(p/'metadata.parquet')
labs  = pd.read_parquet(p/'labels.parquet')['label']

mask = labs.str.strip().str.lower() != 'label'      # drop the 4 stray header rows
feats, meta, labs = feats[mask].reset_index(drop=True), meta[mask].reset_index(drop=True), labs[mask].reset_index(drop=True)
print("removed", (~mask).sum(), "stray rows ->", len(labs), "rows")

feats.to_parquet(p/'features.parquet')
meta.to_parquet(p/'metadata.parquet')
labs.to_frame('label').to_parquet(p/'labels.parquet')
print("rows per class:", labs.value_counts().to_dict())

In [ ]:
import pandas as pd
from pathlib import Path
from driftbench import config, common_core, to_common_core, harmonise_labels, shared_families

feats, labs = {}, {}
for ds in config.DATASETS:
    p = config.INTERIM_DIR / ds
    feats[ds] = pd.read_parquet(p/'features.parquet')
    labs[ds]  = pd.read_parquet(p/'labels.parquet')['label']

# common-core CICFlowMeter features present in BOTH datasets
core = common_core(*feats.values())
print("common-core feature count:", len(core))

# harmonised family labels — confirm nothing falls into 'other'
for ds in config.DATASETS:
    fam = harmonise_labels(labs[ds])
    print(f"\n{ds} families:", fam.value_counts().to_dict())

print("\nshared families (Demo B label space):",
      shared_families(labs[config.DATASETS[0]], labs[config.DATASETS[1]]))

# persist common-core views + family labels for the demos
for ds in config.DATASETS:
    out = config.PROCESSED_DIR / ds; out.mkdir(parents=True, exist_ok=True)
    to_common_core(feats[ds], core).to_parquet(out/'features_core.parquet')
    harmonise_labels(labs[ds]).to_frame('family').to_parquet(out/'labels_family.parquet')
pd.Series(core).to_csv(config.PROCESSED_DIR/'common_core_features.csv', index=False)
print("\nwritten processed/ for both datasets")

In [ ]:
import pandas as pd
from pathlib import Path
from driftbench import config

a = pd.read_parquet(config.INTERIM_DIR/'CSE-CIC-IDS2018'/'features.parquet').columns.tolist()
b = pd.read_parquet(config.INTERIM_DIR/'CIC-DDoS2019'/'features.parquet').columns.tolist()
print("2018 cols:", len(a), "| 2019 cols:", len(b))
print("\nin BOTH:", len(set(a)&set(b)))
print("\n2018-only (first 25):", sorted(set(a)-set(b))[:25])
print("\n2019-only (first 25):", sorted(set(b)-set(a))[:25])

In [ ]:
import pandas as pd, re
from pathlib import Path
from driftbench import config

def canon(col):
    c = col.strip().lower()
    c = re.sub(r'[\s]+', '_', c)
    # unify the two CICFlowMeter abbreviation styles
    repl = [
        (r'\bpkt\b','packet'), (r'\bpkts\b','packets'), (r'\blen\b','length'),
        (r'\bbyts\b','bytes'), (r'\bcnt\b','count'), (r'\btot\b','total'),
        (r'segment_size','seg_size'), (r'seg_size_avg','seg_size_mean'),
        (r'avg_(fwd|bwd)_seg_size', r'\1_seg_size_mean'),
        (r'init_win_bytes_forward','init_fwd_win_bytes'),
        (r'init_win_bytes_backward','init_bwd_win_bytes'),
        (r'act_data_pkt_fwd','fwd_act_data_packets'),
        (r'fwd_act_data_pkts','fwd_act_data_packets'),
        (r'_byts/s','_bytes/s'), (r'flow_byts/s','flow_bytes/s'), (r'flow_pkts/s','flow_packets/s'),
        (r'\bpacket_length\b','packet_len'),  # collapse to one stem
    ]
    for pat,sub in repl:
        c = re.sub(pat, sub, c)
    c = c.replace('packet','pkt').replace('length','len')  # final unify to short stem
    return c

a = pd.read_parquet(config.INTERIM_DIR/'CSE-CIC-IDS2018'/'features.parquet')
b = pd.read_parquet(config.INTERIM_DIR/'CIC-DDoS2019'/'features.parquet')
amap = {c: canon(c) for c in a.columns}
bmap = {c: canon(c) for c in b.columns}
shared = sorted(set(amap.values()) & set(bmap.values()))
print("recovered common-core count:", len(shared))
print(shared)

In [ ]:
import pandas as pd, re
from pathlib import Path
from driftbench import config

def canon(col):
    c = col.strip().lower(); c = re.sub(r'[\s]+','_',c)
    for pat,sub in [(r'\bpkt\b','packet'),(r'\bpkts\b','packets'),(r'\blen\b','length'),
        (r'\bbyts\b','bytes'),(r'\bcnt\b','count'),(r'\btot\b','total'),
        (r'segment_size','seg_size'),(r'seg_size_avg','seg_size_mean'),
        (r'avg_(fwd|bwd)_seg_size',r'\1_seg_size_mean'),
        (r'init_win_bytes_forward','init_fwd_win_bytes'),(r'init_win_bytes_backward','init_bwd_win_bytes'),
        (r'act_data_pkt_fwd','fwd_act_data_packets'),(r'fwd_act_data_pkts','fwd_act_data_packets'),
        (r'_byts/s','_bytes/s'),(r'flow_byts/s','flow_bytes/s'),(r'flow_pkts/s','flow_packets/s'),
        (r'\bpacket_length\b','packet_len')]:
        c = re.sub(pat,sub,c)
    return c.replace('packet','pkt').replace('length','len')

core = ['active_max','active_mean','active_min','active_std','bwd_header_len','bwd_iat_max','bwd_iat_mean','bwd_iat_min','bwd_iat_std','bwd_pkt_len_max','bwd_pkt_len_mean','bwd_pkt_len_min','bwd_pkt_len_std','bwd_pkts/s','bwd_seg_size_mean','cwe_flag_count','down/up_ratio','flow_bytes/s','flow_duration','flow_iat_max','flow_iat_mean','flow_iat_min','flow_iat_std','flow_pkts/s','fwd_act_data_pkts','fwd_header_len','fwd_iat_max','fwd_iat_mean','fwd_iat_min','fwd_iat_std','fwd_pkt_len_max','fwd_pkt_len_mean','fwd_pkt_len_min','fwd_pkt_len_std','fwd_pkts/s','fwd_psh_flags','fwd_seg_size_mean','idle_max','idle_mean','idle_min','idle_std','pkt_len_mean','pkt_len_std','subflow_bwd_pkts','subflow_fwd_pkts']

for ds in config.DATASETS:
    p = config.INTERIM_DIR / ds
    f = pd.read_parquet(p/'features.parquet')
    f = f.rename(columns={c: canon(c) for c in f.columns})
    f = f.loc[:, [c for c in core if c in f.columns]]   # select+order the shared core
    f = f.loc[:, ~f.columns.duplicated()]               # guard against any collision
    out = config.PROCESSED_DIR / ds; out.mkdir(parents=True, exist_ok=True)
    f.to_parquet(out/'features_core.parquet')
    print(ds, "core features:", f.shape)

pd.Series(core).to_csv(config.PROCESSED_DIR/'common_core_features.csv', index=False)
print("rebuilt features_core for both — aligned on", len(core), "features")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, subprocess, os, sys
z = glob.glob("/content/drive/MyDrive/**/drift-conference*.zip", recursive=True)[0]
subprocess.run(["rm","-rf","/content/dc_pkg"], check=True)
os.makedirs("/content/dc_pkg", exist_ok=True)
subprocess.run(["unzip","-q",z,"-d","/content/dc_pkg"], check=True)
subprocess.run(["pip","-q","install","--no-deps","/content/dc_pkg/drift-conference"], check=True)

# purge any stale copy, then import
for m in list(sys.modules):
    if m=="driftbench" or m.startswith("driftbench."):
        del sys.modules[m]
import driftbench as db
print("driftbench:", db.__version__)
from driftbench import config, make_model, compute_metrics, metrics_frame, random_split, chronological_split, leave_one_day_out
print("imports OK")

In [ ]:
import pandas as pd, numpy as np, gc
from sklearn.ensemble import RandomForestClassifier

ds = 'CSE-CIC-IDS2018'
p = config.INTERIM_DIR / ds
X    = pd.read_parquet(p/'features.parquet').astype('float32')   # half the RAM
y    = pd.read_parquet(p/'labels.parquet')['label']
meta = pd.read_parquet(p/'metadata.parquet')
print("X", X.shape, "| meta cols:", list(meta.columns))

def rf(): return RandomForestClassifier(n_estimators=120, max_depth=24, n_jobs=-1,
                                        random_state=0, class_weight='balanced_subsample')
def fit_eval(tr, te):
    m = rf().fit(X.iloc[tr], y.iloc[tr])
    out = compute_metrics(y.iloc[te], m.predict(X.iloc[te]), m.predict_proba(X.iloc[te]), m.classes_)
    del m; gc.collect()
    return out

rows = {}
rs = random_split(len(X), y, seed=0);  rows['random'] = fit_eval(rs['train'], rs['test'])
cs = chronological_split(meta);        rows['chrono'] = fit_eval(cs['train'], cs['test'])
lodo = leave_one_day_out(meta); print("LODO folds:", len(lodo))
fold = [fit_eval(f['train'], f['test']) for _, f in lodo]
rows['lodo'] = {k: float(np.nanmean([fm[k] for fm in fold])) for k in fold[0]}

tbl = metrics_frame(rows)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
tbl.to_csv(config.RESULTS_DIR/f'demoA_{ds}_rf.csv')
print("\n=== Demo A: CSE-CIC-IDS2018, RF ===")
print(tbl[['macro_f1','weighted_f1','mcc','ece']].round(4))

In [ ]:
import pandas as pd
from driftbench import config, chronological_split, leave_one_day_out
p = config.INTERIM_DIR/'CSE-CIC-IDS2018'
y = pd.read_parquet(p/'labels.parquet')['label']
meta = pd.read_parquet(p/'metadata.parquet')

cs = chronological_split(meta)
print("train classes:", sorted(y.iloc[cs['train']].unique()))
print("test classes :", sorted(y.iloc[cs['test']].unique()))
print("overlap       :", len(set(y.iloc[cs['train']].unique()) & set(y.iloc[cs['test']].unique())))

print("\nday distribution:")
ts = pd.to_datetime(meta['timestamp'], errors='coerce')
print(pd.crosstab(ts.dt.date, y).astype(bool).sum(axis=1), "classes per day")

In [ ]:
import pandas as pd
from driftbench import config
meta = pd.read_parquet(config.INTERIM_DIR/'CSE-CIC-IDS2018'/'metadata.parquet')
print(meta['timestamp'].dropna().head(10).tolist())
print("\nunique sample:", meta['timestamp'].dropna().sample(5, random_state=1).tolist())

In [ ]:
import pandas as pd, numpy as np, gc
from sklearn.ensemble import RandomForestClassifier
from driftbench import config, compute_metrics, metrics_frame, random_split

p = config.INTERIM_DIR/'CSE-CIC-IDS2018'
X = pd.read_parquet(p/'features.parquet').astype('float32')
y = pd.read_parquet(p/'labels.parquet')['label'].reset_index(drop=True)
meta = pd.read_parquet(p/'metadata.parquet')
ts = pd.to_datetime(meta['timestamp'], dayfirst=True, errors='coerce')   # <-- the fix
print("days recovered:", sorted(ts.dt.date.dropna().unique()))

def rf(): return RandomForestClassifier(n_estimators=120, max_depth=24, n_jobs=-1,
                                        random_state=0, class_weight='balanced_subsample')
def fit_eval(tr, te):
    m = rf().fit(X.iloc[tr], y.iloc[tr])
    out = compute_metrics(y.iloc[te], m.predict(X.iloc[te]), m.predict_proba(X.iloc[te]), m.classes_)
    del m; gc.collect(); return out

# stratified temporal split: per class, earliest 60% -> train, latest 40% -> test
order = np.argsort(ts.values, kind='stable')
rank_in_class = pd.Series(order).groupby(y.iloc[order].values).cumcount()
# build train/test index by time-rank within each class
tr_idx, te_idx = [], []
for cls in y.unique():
    idx_c = order[y.iloc[order].values == cls]
    cut = int(len(idx_c)*0.6)
    tr_idx += list(idx_c[:cut]); te_idx += list(idx_c[cut:])
tr_idx, te_idx = np.array(tr_idx), np.array(te_idx)

# plain chronological (time-blind) split for the secondary point
n=len(X); a=int(n*0.6)
chrono_tr, chrono_te = order[:a], order[a:]

rows = {}
rs = random_split(len(X), y, seed=0);           rows['random']        = fit_eval(rs['train'], rs['test'])
rows['temporal_stratified'] = fit_eval(tr_idx, te_idx)
rows['chronological_blind'] = fit_eval(chrono_tr, chrono_te)

tbl = metrics_frame(rows)
tbl.to_csv(config.RESULTS_DIR/'demoA_CSE-CIC-IDS2018_rf.csv')
print("\n=== Demo A (corrected dates) ===")
print(tbl[['macro_f1','weighted_f1','mcc','ece']].round(4))
print("\ntemporal-split class overlap:", len(set(y.iloc[tr_idx]) & set(y.iloc[te_idx])), "of", y.nunique())

In [ ]:
import os
from pathlib import Path
RESULTS = Path('/content/drive/MyDrive/drift-conference/results')
RESULTS.mkdir(parents=True, exist_ok=True)
os.environ['DRIFT_REPO_ROOT'] = '/content/drive/MyDrive/drift-conference'  # so config.RESULTS_DIR -> Drive
print("results will save to:", RESULTS)

In [ ]:
import pandas as pd
from pathlib import Path
R = Path('/content/drive/MyDrive/drift-conference/results'); R.mkdir(parents=True, exist_ok=True)
# tbl is still in memory from the last cell
tbl.to_csv(R/'demoA_CSE-CIC-IDS2018_rf.csv')
print("saved:", (R/'demoA_CSE-CIC-IDS2018_rf.csv').exists())
print(pd.read_csv(R/'demoA_CSE-CIC-IDS2018_rf.csv', index_col=0).round(4))

In [ ]:
import pandas as pd, numpy as np, gc
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from driftbench import config, compute_metrics, metrics_frame, random_split

R = Path('/content/drive/MyDrive/drift-conference/results'); R.mkdir(parents=True, exist_ok=True)
p = config.INTERIM_DIR/'CSE-CIC-IDS2018'
X = pd.read_parquet(p/'features.parquet').astype('float32')
y = pd.read_parquet(p/'labels.parquet')['label'].reset_index(drop=True)
ts = pd.to_datetime(pd.read_parquet(p/'metadata.parquet')['timestamp'], dayfirst=True, errors='coerce')
order = np.argsort(ts.values, kind='stable')

# build the three split index sets once
rs = random_split(len(X), y, seed=0)
tr_idx, te_idx = [], []
for cls in y.unique():
    idx_c = order[y.iloc[order].values == cls]; cut = int(len(idx_c)*0.6)
    tr_idx += list(idx_c[:cut]); te_idx += list(idx_c[cut:])
tr_idx, te_idx = np.array(tr_idx), np.array(te_idx)
a = int(len(X)*0.6); chrono_tr, chrono_te = order[:a], order[a:]
splits = {'random': (rs['train'], rs['test']),
          'temporal_stratified': (tr_idx, te_idx),
          'chronological_blind': (chrono_tr, chrono_te)}

def model(name):
    if name=='rf':   return RandomForestClassifier(n_estimators=120, max_depth=24, n_jobs=-1, random_state=0, class_weight='balanced_subsample')
    if name=='lgbm':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, class_weight='balanced', n_jobs=-1, random_state=0, verbose=-1)
    if name=='mlp':  return Pipeline([('s',StandardScaler()),('m',MLPClassifier(hidden_layer_sizes=(128,64), max_iter=150, random_state=0))])

import warnings; warnings.filterwarnings('ignore')
for name in ['lgbm','mlp']:                       # rf already saved
    rows = {}
    for split, (tr, te) in splits.items():
        mdl = model(name).fit(X.iloc[tr], y.iloc[tr])
        rows[split] = compute_metrics(y.iloc[te], mdl.predict(X.iloc[te]), mdl.predict_proba(X.iloc[te]), mdl.classes_)
        del mdl; gc.collect()
    tbl = metrics_frame(rows)
    tbl.to_csv(R/f'demoA_CSE-CIC-IDS2018_{name}.csv')
    print(f"\n=== Demo A: {name} ===")
    print(tbl[['macro_f1','weighted_f1','mcc','ece']].round(4))

In [ ]:
import pandas as pd
from pathlib import Path
R = Path('/content/drive/MyDrive/drift-conference/results')
for n in ['rf','lgbm','mlp']:
    f = R/f'demoA_CSE-CIC-IDS2018_{n}.csv'
    print(n, "saved:", f.exists())

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
R = Path('/content/drive/MyDrive/drift-conference/results'); F = R/'figures'; F.mkdir(exist_ok=True)

models = ['rf','lgbm','mlp']; labels=['Random Forest','LightGBM','MLP']
conds = ['random','temporal_stratified','chronological_blind']
cond_lbl = ['Random','Temporal\n(stratified)','Chronological\n(blind)']
colors = ['#4C72B0','#55A868','#C44E52']

data = {m: pd.read_csv(R/f'demoA_CSE-CIC-IDS2018_{m}.csv', index_col=0) for m in models}

def grouped(metric, ylabel, fname, ylim=None):
    fig, ax = plt.subplots(figsize=(7,4.2))
    x = np.arange(len(models)); w=0.26
    for i,c in enumerate(conds):
        vals=[data[m].loc[c,metric] for m in models]
        ax.bar(x+(i-1)*w, vals, w, label=cond_lbl[i], color=colors[i], edgecolor='black', linewidth=0.5)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel); ax.legend(frameon=False, fontsize=9)
    if ylim: ax.set_ylim(*ylim)
    ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
    for s in ['top','right']: ax.spines[s].set_visible(False)
    plt.tight_layout()
    fig.savefig(F/f'{fname}.pdf'); fig.savefig(F/f'{fname}.png', dpi=200)
    plt.show(); print("saved", fname)

grouped('macro_f1','Macro-F1','fig_demoA_macroF1', ylim=(0,1))
grouped('ece','Expected Calibration Error','fig_demoA_ece', ylim=(0,1))

In [ ]:
import pandas as pd, numpy as np, gc, warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from driftbench import config, compute_metrics, metrics_frame
warnings.filterwarnings('ignore')

R = Path('/content/drive/MyDrive/drift-conference/results'); R.mkdir(parents=True, exist_ok=True)
A, B = 'CSE-CIC-IDS2018', 'CIC-DDoS2019'

# load common-core features + family labels, restrict to shared families {benign, ddos}
def load(ds):
    pf = config.PROCESSED_DIR/ds
    X = pd.read_parquet(pf/'features_core.parquet').astype('float32')
    fam = pd.read_parquet(pf/'labels_family.parquet')['family'].astype(str).reset_index(drop=True)
    return X.reset_index(drop=True), fam

Xa, ya = load(A); Xb, yb = load(B)
shared = ['benign','ddos']
ma = ya.isin(shared).values; mb = yb.isin(shared).values
Xa, ya = Xa[ma].reset_index(drop=True), ya[ma].reset_index(drop=True)
Xb, yb = Xb[mb].reset_index(drop=True), yb[mb].reset_index(drop=True)
# align column order
Xb = Xb[Xa.columns]
print("train(2018):", Xa.shape, dict(ya.value_counts()))
print("test (2019):", Xb.shape, dict(yb.value_counts()))

# 2019 time order for the calibration buffer (use its metadata timestamp)
tb = pd.to_datetime(pd.read_parquet(config.INTERIM_DIR/B/'metadata.parquet')['timestamp'], dayfirst=True, errors='coerce')
tb = tb[mb].reset_index(drop=True)
order = np.argsort(tb.values, kind='stable')

base = RandomForestClassifier(n_estimators=120, max_depth=24, n_jobs=-1, random_state=0,
                              class_weight='balanced_subsample').fit(Xa, ya)

rows = {}
# zero-shot transfer: train 2018 -> test all of 2019
rows['zero_shot'] = compute_metrics(yb, base.predict(Xb), base.predict_proba(Xb), base.classes_)

# calibration-buffer sweep: recalibrate on earliest frac of 2019, test on the rest
for frac in (0.01, 0.05, 0.10):
    k = max(int(len(order)*frac), 1)
    buf, rest = order[:k], order[k:]
    try:
        from sklearn.frozen import FrozenEstimator
        cal = CalibratedClassifierCV(FrozenEstimator(base), method='sigmoid')
    except ImportError:
        cal = CalibratedClassifierCV(base, method='sigmoid', cv='prefit')
    cal.fit(Xb.iloc[buf], yb.iloc[buf])
    rows[f'buffer_{int(frac*100)}pct'] = compute_metrics(
        yb.iloc[rest], cal.predict(Xb.iloc[rest]), cal.predict_proba(Xb.iloc[rest]), cal.classes_)
    gc.collect()

tbl = metrics_frame(rows)
tbl.to_csv(R/'demoB_transfer_rf.csv')
print("\n=== Demo B: 2018 -> 2019 transfer (RF) ===")
print(tbl[['macro_f1','weighted_f1','mcc','brier','ece']].round(4))

In [ ]:
import pandas as pd
from pathlib import Path
R = Path('/content/drive/MyDrive/drift-conference/results')
print("demoB saved:", (R/'demoB_transfer_rf.csv').exists())
print(pd.read_csv(R/'demoB_transfer_rf.csv', index_col=0).round(4))

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
R = Path('/content/drive/MyDrive/drift-conference/results'); F = R/'figures'; F.mkdir(exist_ok=True)
d = pd.read_csv(R/'demoB_transfer_rf.csv', index_col=0)

x = [0,1,5,10]
ece = [d.loc['zero_shot','ece'], d.loc['buffer_1pct','ece'], d.loc['buffer_5pct','ece'], d.loc['buffer_10pct','ece']]
mcc = [d.loc['zero_shot','mcc'], d.loc['buffer_1pct','mcc'], d.loc['buffer_5pct','mcc'], d.loc['buffer_10pct','mcc']]

fig, ax1 = plt.subplots(figsize=(7,4.2))
l1,=ax1.plot(x, ece, 'o-', color='#C44E52', lw=2, ms=7, label='ECE (calibration)')
ax1.set_xlabel('Target calibration buffer (% of 2019, time-ordered)')
ax1.set_ylabel('Expected Calibration Error', color='#C44E52'); ax1.tick_params(axis='y', labelcolor='#C44E52')
ax1.set_ylim(0,1)
ax2 = ax1.twinx()
l2,=ax2.plot(x, mcc, 's--', color='#4C72B0', lw=2, ms=7, label='MCC (discrimination)')
ax2.set_ylabel('Matthews Correlation Coefficient', color='#4C72B0'); ax2.tick_params(axis='y', labelcolor='#4C72B0')
ax2.set_ylim(-0.1,1)
ax2.axhline(0, color='#4C72B0', ls=':', alpha=0.4)
ax1.set_xticks(x); ax1.grid(alpha=0.3); ax1.set_axisbelow(True)
for s in ['top']: ax1.spines[s].set_visible(False); ax2.spines[s].set_visible(False)
ax1.legend(handles=[l1,l2], frameon=False, loc='center right', fontsize=9)
plt.title('2018→2019 transfer: recalibration repairs confidence, not discrimination', fontsize=10)
plt.tight_layout()
fig.savefig(F/'fig_demoB_recovery.pdf'); fig.savefig(F/'fig_demoB_recovery.png', dpi=200)
plt.show(); print("saved fig_demoB_recovery")

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from driftbench import config, feature_drift_report, label_psi, psi_band

R = Path('/content/drive/MyDrive/drift-conference/results'); R.mkdir(parents=True, exist_ok=True)
A, B = 'CSE-CIC-IDS2018', 'CIC-DDoS2019'

Xa = pd.read_parquet(config.PROCESSED_DIR/A/'features_core.parquet').astype('float32')
Xb = pd.read_parquet(config.PROCESSED_DIR/B/'features_core.parquet').astype('float32')[Xa.columns]

rep = feature_drift_report(Xa, Xb, Xa.columns)        # PSI + KS per feature, ranked by PSI
rep['psi_band'] = rep['psi'].map(psi_band)
rep.to_csv(R/'drift_cross_2018_2019.csv', index=False)

print("=== Cross-dataset drift (2018 vs 2019), top 15 by PSI ===")
print(rep.head(15).round(4).to_string(index=False))
print("\nPSI band counts:", rep['psi_band'].value_counts().to_dict())
print("features with significant drift (PSI>=0.25):", int((rep['psi']>=0.25).sum()), "of", len(rep))

# label-distribution drift (benign/ddos prevalence shift between datasets)
fa = pd.read_parquet(config.PROCESSED_DIR/A/'labels_family.parquet')['family']
fb = pd.read_parquet(config.PROCESSED_DIR/B/'labels_family.parquet')['family']
print("\nlabel-distribution PSI (2018 vs 2019):", round(label_psi(fa, fb), 4))

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
R = Path('/content/drive/MyDrive/drift-conference/results'); F = R/'figures'; F.mkdir(exist_ok=True)
rep = pd.read_csv(R/'drift_cross_2018_2019.csv')
print("drift csv saved:", (R/'drift_cross_2018_2019.csv').exists())

top = rep.nlargest(15,'ks').iloc[::-1]   # rank by KS (bounded, honest), ascending for barh
fig, ax = plt.subplots(figsize=(7,5))
ax.barh(top['feature'], top['ks'], color='#C44E52', edgecolor='black', linewidth=0.4)
ax.set_xlabel('Kolmogorov–Smirnov statistic (2018 vs 2019)')
ax.set_xlim(0,1); ax.grid(axis='x', alpha=0.3); ax.set_axisbelow(True)
for s in ['top','right']: ax.spines[s].set_visible(False)
ax.set_title(f'Cross-dataset feature drift: {int((rep.psi>=0.25).sum())}/{len(rep)} features significant (PSI≥0.25)', fontsize=9.5)
plt.tight_layout()
fig.savefig(F/'fig_drift_cross.pdf'); fig.savefig(F/'fig_drift_cross.png', dpi=200)
plt.show(); print("saved fig_drift_cross")

In [ ]:
import pandas as pd, numpy as np, gc, warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, matthews_corrcoef
from driftbench import config, random_split
warnings.filterwarnings('ignore')

R = Path('/content/drive/MyDrive/drift-conference/results')
rng = np.random.default_rng(0)
def ci(y_true, y_pred, fn, B=1000):
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); n=len(y_true)
    pt=fn(y_true,y_pred)
    s=np.array([fn(y_true[i],y_pred[i]) for i in (rng.integers(0,n,n) for _ in range(B))])
    return pt, np.percentile(s,2.5), np.percentile(s,97.5)
macro=lambda a,b: f1_score(a,b,average='macro',zero_division=0)

# --- Demo A: 2018, random vs chronological_blind (RF) ---
p=config.INTERIM_DIR/'CSE-CIC-IDS2018'
X=pd.read_parquet(p/'features.parquet').astype('float32')
y=pd.read_parquet(p/'labels.parquet')['label'].reset_index(drop=True)
ts=pd.to_datetime(pd.read_parquet(p/'metadata.parquet')['timestamp'],dayfirst=True,errors='coerce')
order=np.argsort(ts.values,kind='stable'); a=int(len(X)*0.6)
def rf(): return RandomForestClassifier(n_estimators=120,max_depth=24,n_jobs=-1,random_state=0,class_weight='balanced_subsample')
out=[]
rs=random_split(len(X),y,seed=0); m=rf().fit(X.iloc[rs['train']],y.iloc[rs['train']]); pr=m.predict(X.iloc[rs['test']])
out.append(('demoA_random_macroF1',)+ci(y.iloc[rs['test']],pr,macro)); del m; gc.collect()
m=rf().fit(X.iloc[order[:a]],y.iloc[order[:a]]); pc=m.predict(X.iloc[order[a:]])
out.append(('demoA_blind_macroF1',)+ci(y.iloc[order[a:]],pc,macro)); del m; gc.collect()

# --- Demo B: zero-shot transfer MCC ---
Xa=pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'features_core.parquet').astype('float32')
ya=pd.read_parquet(config.PROCESSED_DIR/'CSE-CIC-IDS2018'/'labels_family.parquet')['family'].astype(str).reset_index(drop=True)
Xb=pd.read_parquet(config.PROCESSED_DIR/'CIC-DDoS2019'/'features_core.parquet').astype('float32')
yb=pd.read_parquet(config.PROCESSED_DIR/'CIC-DDoS2019'/'labels_family.parquet')['family'].astype(str).reset_index(drop=True)
sh=['benign','ddos']; ma=ya.isin(sh).values; mb=yb.isin(sh).values
Xa,ya=Xa[ma],ya[ma]; Xb,yb=Xb[mb].reset_index(drop=True)[Xa.columns],yb[mb].reset_index(drop=True)
m=rf().fit(Xa,ya); pb=m.predict(Xb)
out.append(('demoB_zeroshot_mcc',)+ci(yb,pb,matthews_corrcoef)); del m; gc.collect()

res=pd.DataFrame(out,columns=['metric','point','ci_low','ci_high'])
res.to_csv(R/'bootstrap_cis.csv',index=False)
print(res.round(4).to_string(index=False))

In [ ]:
import psutil, os
m = psutil.virtual_memory()
print(f"RAM used {m.percent}%  ({m.used/1e9:.1f} / {m.total/1e9:.1f} GB)")

In [ ]:
import glob
for f in glob.glob('/content/drive/MyDrive/**/*.ipynb', recursive=True):
    print(f)

In [ ]:
import json, glob, os

candidates = glob.glob('/content/drive/MyDrive/Colab Notebooks/*.ipynb')
keywords = ['driftbench', 'demoA', 'demoB', 'demo_a', 'demo_b',
            'CIC-DDoS2019', 'DDoS2019', 'PSI', 'concept drift', 'common_core']

for path in candidates:
    try:
        nb = json.load(open(path))
        text = ' '.join(''.join(c.get('source', [])) for c in nb.get('cells', []))
        hits = [k for k in keywords if k.lower() in text.lower()]
        if hits:
            print(f"{os.path.basename(path):45s} -> {hits}")
    except Exception as e:
        print(f"{os.path.basename(path):45s} -> (could not read: {e})")

In [ ]:
import os, shutil

dest = '/content/drive/MyDrive/Drift_Research/drift-conference/notebooks'
os.makedirs(dest, exist_ok=True)

moves = {
    '/content/drive/MyDrive/Colab Notebooks/drift_notebook1.ipynb': f'{dest}/01_drift_experiments.ipynb',
    '/content/drive/MyDrive/Colab Notebooks/drift_notebook2.ipynb': f'{dest}/02_drift_experiments.ipynb',
}

for src, dst in moves.items():
    assert os.path.exists(src), f"NOT FOUND: {src}"   # stops safely if a name is wrong
    assert not os.path.exists(dst), f"TARGET EXISTS, would overwrite: {dst}"  # stops if dest already there
    shutil.move(src, dst)
    print("moved ->", dst)

print("\nContents of", dest, ":")
for f in sorted(os.listdir(dest)):
    print("  ", f)

In [ ]:
import os

for root in ['/content/drive/MyDrive/drift-conference',
             '/content/drive/MyDrive/Drift_Research/drift-conference']:
    print("="*60)
    print(root, "->", "EXISTS" if os.path.exists(root) else "DOES NOT EXIST")
    if os.path.exists(root):
        for dirpath, dirnames, filenames in os.walk(root):
            depth = dirpath.replace(root, '').count('/')
            indent = '  ' * depth
            print(f"{indent}{os.path.basename(dirpath)}/")
            for fn in sorted(filenames)[:10]:
                print(f"{indent}  {fn}")
            if len(filenames) > 10:
                print(f"{indent}  ... (+{len(filenames)-10} more)")

In [ ]:
import os, shutil

home_nb = '/content/drive/MyDrive/drift-conference/notebooks'   # the real project root
stray   = '/content/drive/MyDrive/Drift_Research/drift-conference/notebooks'
os.makedirs(home_nb, exist_ok=True)

# normalize both notebooks to consistent names, into the real notebooks/ folder
moves = {
    f'{stray}/drift_notebook1.ipynb':      f'{home_nb}/01_drift_experiments.ipynb',
    f'{stray}/02_drift_experiments.ipynb': f'{home_nb}/02_drift_experiments.ipynb',
}
for src, dst in moves.items():
    if os.path.exists(src):
        assert not os.path.exists(dst), f"TARGET EXISTS: {dst}"
        shutil.move(src, dst)
        print("moved ->", dst)
    else:
        print("skip (not found):", src)

# remove the now-empty stray Drift_Research tree
stray_root = '/content/drive/MyDrive/Drift_Research'
leftover = [f for f in os.listdir(stray) if not f.startswith('.')] if os.path.exists(stray) else []
if not leftover:
    shutil.rmtree(stray_root, ignore_errors=True)
    print("removed stray tree:", stray_root)
else:
    print("NOT removing — stray still has files:", leftover)

print("\nFinal notebooks/ contents:")
for f in sorted(os.listdir(home_nb)):
    print("  ", f)